# Preamble

In [ ]:
import warnings
import logging
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from mlflow.client import MlflowClient
from  sklearn.metrics import mean_absolute_error, mean_squared_error, \
    root_mean_squared_log_error
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse 
import joblib, tempfile

os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

2
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Enable PyTorch Lightning autologging
mlflow.pytorch.autolog(
    log_every_n_epoch=1,
    log_models=True,
    disable=False
)

experiment_name = "store_sales_forecasting"


In [ ]:
# Callable function to compute RMSLE using utilsforecast evaluate function
def rmsle(df, models, id_col='unique_id', target_col='y'):
    out = {id_col: [], 'metric': []}
    for uid, g in df.groupby(id_col):
        out[id_col].append(uid)
        out['metric'].append('rmsle')
        for m in models:
            out.setdefault(m, []).append(
                root_mean_squared_log_error(g[target_col], g[m])
            )
    return pd.DataFrame(out)

In [ ]:
from pathlib import Path

# Resolve project root dynamically
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

db_path = project_root / "mlflow.db"
artifact_path = project_root / "mlartifacts"

# Set tracking URI to SQLite
mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")

# Create experiment with explicit artifact destination
client = MlflowClient()

experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_path.as_uri()
    )

mlflow.set_experiment(experiment_name)

In [ ]:
kagglehub.login()

In [ ]:
competition_name = "store-sales-time-series-forecasting"
path = kagglehub.competition_download(competition_name)

In [ ]:
type(path)
print("".join([path, '/train.csv']))

# Data Ingestion

In [ ]:
train_path = "".join([path, '/train.csv'])

In [ ]:
train = pd.read_csv("".join([path, '/train.csv']), parse_dates=['date'])
stores = pd.read_csv("".join([path, '/stores.csv']))

# Exploratory Data Analysis

## Dataset Overview

In [ ]:
print(train.head(5), "\n", train.tail(5))

In [ ]:
print(train.columns)

In [ ]:
train.describe()

In [ ]:
train.info()

In [ ]:
print(f"Number of categories or families and names: {train['family'].nunique()} \n {train['family'].unique()[:]}")


## Feature Engineering

In [ ]:
train['unique_id'] = train['store_nbr'].astype('str') + '_' + train['family']

In [ ]:
print(train.columns)
print(stores.columns)

In [ ]:
train.drop(columns='unique_id')

In [ ]:
print(f"Unique Time Series Count: {train['unique_id'].nunique()}")

In [ ]:
print(f"Date rage: {train['date'].min()} - {train['date'].max()}")

## Forecasting DataFrame

In [ ]:
df = train[["unique_id", "date", "sales"]].rename(columns={"date": "ds", "sales": "y"})

In [ ]:
df

In [ ]:
zero_pct = (df['y'] == 0).mean()
print(f"Zero sales percentage: {zero_pct:.1%}")

## Time Series Patterns

In [ ]:
# Plot a few series to see patterns
fix, axes = plt.subplots(10, 1, figsize=(14, 20))
sample_ids = df['unique_id'].sample(n=10, random_state=1)
for uid, ax in zip(sample_ids, axes):
    subset = df[df['unique_id'] == uid]
    ax.plot(subset['ds'], subset['y'])
    ax.set_title(uid)
plt.tight_layout()
plt.savefig('eda_sample_series.png')
plt.show()

In [ ]:
# Top 10 intermittent series
intermittent = df.groupby('unique_id')['y'].apply(lambda x: (x == 0).mean())
print(f"\n Top 10 most intermittent series:")
print(intermittent.sort_values(ascending=False).head(10))

# StatsForecast Baseline

## Data Subsetting and Train-Test Split

In [ ]:
print(df['unique_id'].nunique())

In [ ]:
# SUBSET first
subset_categories = ['1_BEVERAGES', '34_PREPARED FOODS', '8_CLEANING', '33_DAIRY', '10_GROCERY I']

In [ ]:
df_sample = df[df['unique_id'].isin(subset_categories)].copy()

In [ ]:
# Split: last 16 days as test (Kaggle uses 16-day horizon)
# df_sample['ds'] = pd.to_datetime(df_sample['ds']).dt.normalize()

max_date_per_series = df_sample.groupby('unique_id', as_index=False)['ds'].transform('max')

cutoff = max_date_per_series - pd.Timedelta(days=16)
train_df = df_sample[df_sample['ds'] <= cutoff]
test_df = df_sample[df_sample['ds'] > cutoff]

In [ ]:
print(train_df['ds'].max(), test_df['ds'].min())

In [ ]:
train_df.shape

In [ ]:
train_df.groupby('unique_id').mean()

In [ ]:
test_df.shape

## Model Training and Forecast

In [ ]:
# Function to evaluate each model for each series

def evaluate_models(eval_df: pd.DataFrame, model_cols: list) -> pd.DataFrame:
    results = []
    for uid in eval_df['unique_id'].unique():
        mask = eval_df['unique_id'] == uid
        for model in model_cols: 
            mae = mean_absolute_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, model])
            rsme = np.sqrt(mean_squared_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, model]))
            rmsle = root_mean_squared_log_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, model])
            results.append({'unique_id': uid, 'model': model, 'MAE': mae, 'RMSE': rsme, 'RMSLE': rmsle})


    return pd.DataFrame(results)

In [ ]:
# Define models
from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA,
    AutoETS,
    AutoCES,
    CrostonOptimized, # For intermittent demand
    SeasonalNaive, 
    HistoricAverage
)
from statsforecast.utils import ConformalIntervals # This allow to calculate the Confidence intervals

# Parameters
horizon = 16
season_length = 7
freq = 'D' # Daily data
conf_level = 90
n_windows = 2

intervals = ConformalIntervals(h=horizon, n_windows=n_windows) # TODO: How this works.

models = [
    AutoARIMA(season_length=season_length, prediction_intervals=intervals),
    AutoETS(season_length=season_length, prediction_intervals=intervals),
    AutoCES(season_length=season_length, prediction_intervals=intervals),
    CrostonOptimized(prediction_intervals=intervals), # Intermittent demand
    SeasonalNaive(season_length=season_length, prediction_intervals=intervals), 
    HistoricAverage(prediction_intervals=intervals)
]

model_names = [type(model).__name__ for model in models]

params_to_log = {
    "forecast_horizon": horizon,
    "forecast_conf_level": conf_level,
    "sf_season_length": season_length,
    "sf_models": model_names,
    "sf_freq": freq,
    "interval_type": "Conformal",
    "interval_n_windows": n_windows,
}

# Feature Set
standard_cols = ['unique_id', 'ds', 'y']
exogenous_features = [col for col in train_df.columns if col not in standard_cols]

import time

with mlflow.start_run(run_name="statsforecast/baseline/2026-08-26") as run:
    start_time = time.perf_counter()
    # Log dataset metadata
    dataset = mlflow.data.from_pandas(
        train_df, 
        source=f"/home/{os.getenv('USER')}/.cache/kagglehub/competitions/{competition_name}/train.csv",
        name='training_dataset'
    )

    mlflow.log_input(dataset, context="training")
    print("Dataset metadata logged successfully.")

    # Log architecture parameters
    mlflow.log_params(params_to_log)
    print("Parameters logged successfully.")


    # Log feature set
    mlflow.log_dict({"baseline_features": standard_cols}, "feature_sets/features.json")
    print("Feature set logged successfully.")


    # Fit and forecast
    sf = StatsForecast(
        models=models, 
        freq=freq,
        n_jobs=-1, # Parallel
    )


    sf.save(path="statsforecast_models")
    mlflow.log_artifacts("statsforecast_models", 
                         artifact_path = "statsforecast_models")

    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as tmp_file:
        joblib.dump(sf, tmp_file.name)
        mlflow.log_artifact(tmp_file.name, "model")
        print(f"StatsForecast model saved and logged to MLflow at {tmp_file.name}")

    # Use .forecast() for speed (no fitted values stored)
    forecasts = sf.forecast(df=train_df, h=horizon, level=[conf_level])
    print("Generated StatsForecasts.")

    end_time = time.perf_counter()
    run_time = end_time - start_time
    mlflow.log_metric("training_time_in_seconds", run_time)


    # Models Evaluation
    eval_df = test_df.merge(forecasts.reset_index(), on=['unique_id', 'ds'])
    model_cols = [c for c in forecasts.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]
    result_df = evaluate_models(eval_df=eval_df, model_cols=model_cols)

    for _, row in result_df.iterrows():
        mlflow.log_metrics({
            f"{row['model']}_MAE": row['MAE'],
            f"{row['model']}_RMSE": row['RMSE'],
            f"{row['model']}_RMSLE": row['RMSLE']
        })

    

In [ ]:
print(forecasts.head(6))

## Evaluation

In [ ]:
test_df.columns

In [ ]:
# Evaluation
eval_df = test_df.merge(forecasts.reset_index(), on=['unique_id', 'ds'])
model_cols = [c for c in forecasts.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]

In [ ]:
model_cols

In [ ]:

# Evaluate each model for each series
mask = eval_df['unique_id'] == '8_CLEANING'
print(mean_absolute_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, 'AutoARIMA']))

In [ ]:

def print_evaluation_results(results_df: pd.DataFrame):
    print("\n=== Model Comparison ===")
    print(results_df.pivot_table(index='model', values=['MAE', 'RMSE', 'RMSLE'], aggfunc='mean').round(2))

    best_mae = results_df.loc[results_df.groupby('unique_id')['MAE'].idxmin()]
    print("\n=== Best Model per Series (MAE) ===")
    print(best_mae[['unique_id', 'model', 'MAE']])

    best_rmse = results_df.loc[results_df.groupby('unique_id')['RMSE'].idxmin()]
    print("\n=== Best Model per Series (RMSE) ===")
    print(best_rmse[['unique_id', 'model', 'RMSE']])

    best_rmsle = results_df.loc[results_df.groupby('unique_id')['RMSLE'].idxmin()]
    print("\n=== Best Model per Series (RMSLE) ===")
    print(best_rmsle[['unique_id', 'model', 'RMSLE']])

In [ ]:
results_df = evaluate_models(eval_df, model_cols)
print_evaluation_results(results_df)

## Cross-Validation

In [ ]:
# StatForecast has built-in time series cross-validation

# Parameteers for cross-validation
horizon = 16
step_size = 16
n_windows = 3
conf_level = 90

params_to_log_cv = {
    "forecast_horizon": horizon,
    "sf_models": model_names,
    "sf_freq": "D",
    "step_size": step_size,
    "conf_level": conf_level,
    "interval_n_windows": n_windows,
}

# Feature Set
standard_cols = ['unique_id', 'ds', 'y']

with mlflow.start_run(run_name="statsforecast/crossval_baseline/2026-08-26") as run:

    # Log dataset metadata
    dataset = mlflow.data.from_pandas(
        df_sample, 
        source=f"/home/{os.getenv('USER')}/.cache/kaggle/competitions/{competition_name}/train.csv",
        name='crossval_dataset_baseline'
    )

    mlflow.log_input(dataset, context="crossval")
    print("Dataset metadata logged successfully.")

    # Log architecture parameters
    mlflow.log_params(params_to_log_cv)
    print("Parameters logged successfully.")


    # Log feature set
    mlflow.log_dict({"baseline_features": standard_cols}, "feature_sets/baseline_features.json")
    print("Feature set logged successfully.")
    
    crossval = sf.cross_validation(
        df=df_sample,
        h=horizon,  # Forecast horizon 
        step_size=step_size, # Non-overlapping windows
        n_windows=n_windows, # 3 folds
        level=[conf_level], # Confidence intervals
        n_jobs=-1, # Parallel
    )

    print("Cross-validation completed successfully.")

    all_folds_metrics = []

    # Log cross-validation metrics per fold
    for fold_idx, (cutoff, fold_df) in enumerate(crossval.groupby('cutoff')):
        result = evaluate(fold_df.drop(columns=['cutoff']),
                          metrics=[mae, rmse, rmsle]
        )

        mean_metrics = result.drop(columns=['unique_id']).groupby('metric').mean()

        fold_dict = {}

        for metric_name, row in mean_metrics.iterrows():
            for model_name, value in row.items():
                key = f"{model_name}_{metric_name}"
                fold_dict[key] = value
                
        mlflow.log_metrics(fold_dict, step=fold_idx)

        all_folds_metrics.append(fold_dict)


    cv_results_df = pd.DataFrame(all_folds_metrics)

    aggregated_metrics = {}

    for model in cv_results_df.columns:
        aggregated_metrics[f"{model}_cv_mean"]= cv_results_df[model].mean()
        aggregated_metrics[f"{model}_cv_std"] = cv_results_df[model].std()

    mlflow.log_metrics(aggregated_metrics)

    print("Logged individual fold metrics and aggregated totals of the CV.")
    print(crossval.head())

In [ ]:
print("\n=== MAE ===")
for model in model_cols:
    mae = mean_absolute_error(crossval['y'], crossval[model])
    print(f"{model}: CV MAE = {mae:.2f}")

print("\n=== RMSE ===")
for model in model_cols:
    rmse = np.sqrt(mean_squared_error(crossval['y'], crossval[model]))
    print(f"{model}: CV RMSE = {rmse:.2f}")

print("\n=== RMSLE ===")
for model in model_cols:
    rmsle = root_mean_squared_log_error(crossval['y'], crossval[model])
    print(f"{model}: CV RMSLE = {rmsle:.2f}")


## Forecast Visualization

In [ ]:
# Plot forecasts vs actual for 2-3 series
fig, axes = plt.subplots(len(subset_categories[:3]), 1, figsize=(14,10))
for ax, uid in zip(axes, subset_categories[:3]):
    # Historical
    hist = train_df[train_df['unique_id'] == uid].tail(60)
    ax.plot(hist['ds'], hist['y'], label='Historical', color='black')

    # Actuals in test period
    actual = test_df[test_df['unique_id'] == uid]
    actual = pd.concat([train_df[train_df['unique_id'] == uid].tail(1), actual])
    ax.plot(actual['ds'], actual['y'], label='Actual', color='green', linewidth=2)

    # Forecast (best 2 models)
    fc = forecasts.reset_index()
    fc_uid = fc[fc['unique_id'] == uid]
    ax.plot(fc_uid['ds'], fc_uid['AutoARIMA'], label='AutoARIMA', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['AutoETS'], label='AutoETS', linestyle='--')

    # Confidence interval
    if 'AutoARIMA-lo-90' in fc_uid.columns:
        ax.fill_between(fc_uid['ds'], fc_uid['AutoARIMA-lo-90'],
                        fc_uid['AutoARIMA-hi-90'], alpha=0.2)
    ax.set_title(uid)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150)
plt.show()

# Stretch Goals (if time allows)

- [x] Add oil.csv as exogenous variable to AutoARIMA
- [x] Try NeuralForecast: NHITS, PatchTST on same data
- [x] Add MLflow tracking for experiment logging
- [ ] Hierarchical forecasting: store-level → family-level reconciliation

# Exogenous Variables: Oil Price

## Data Loading

In [ ]:
oil_df = pd.read_csv(path + '/oil.csv', parse_dates=['date'])

In [ ]:
print(f"{oil_df.head(5)}", f"\n {oil_df.tail(5)}")

## Missing Values

In [ ]:
oil_df.isna().sum()

In [ ]:
oil_df = oil_df.ffill()


In [ ]:
oil_df.isna().sum()

In [ ]:
oil_df = oil_df.rename(columns={'dcoilwtico': 'oil_price'})

## Merge with Training and Test Data

In [ ]:
exo_train_df = train_df.copy().merge(oil_df, right_on='date', left_on='ds', how='left').drop(columns='date')

In [ ]:
exo_train_df.isna().sum()

In [ ]:
exo_test_df = test_df.copy().merge(oil_df, right_on='date', left_on='ds', how='left').drop(columns='date')

In [ ]:
exo_test_df.isna().sum()

In [ ]:
exo_test_df[exo_test_df['oil_price'].isna()]

In [ ]:
# Front and back fill with valid values after left join.
exo_test_df['oil_price'] = exo_test_df['oil_price'].ffill().bfill()
exo_train_df['oil_price'] = exo_train_df['oil_price'].ffill().bfill()

In [ ]:
exo_train_df.isna().sum()

In [ ]:
print(exo_test_df.shape, test_df.shape)
print(exo_test_df.columns)

## Model Training and Forecast

In [ ]:


# Parameters
horizon = 16
season_length = 7
freq = 'D' # Daily data
conf_level = 90
n_windows = 2

intervals = ConformalIntervals(h=horizon, n_windows=n_windows) # TODO: How this works.

models = [
    AutoARIMA(season_length=season_length, prediction_intervals=intervals),
    AutoETS(season_length=season_length, prediction_intervals=intervals),
    AutoCES(season_length=season_length, prediction_intervals=intervals),
    CrostonOptimized(prediction_intervals=intervals), # Intermittent demand
    SeasonalNaive(season_length=season_length, prediction_intervals=intervals), 
    HistoricAverage(prediction_intervals=intervals)
]

model_names = [type(model).__name__ for model in models]

params_to_log = {
    "forecast_horizon": horizon,
    "forecast_conf_level": conf_level,
    "sf_season_length": season_length,
    "sf_models": model_names,
    "sf_freq": freq,
    "interval_type": "Conformal",
    "interval_n_windows": n_windows,

} 

sources = [
    f"/home/{os.getenv('USER')}/.cache/kagglehub/competitions/{competition_name}/train.csv",
    f"/home/{os.getenv('USER')}/.cache/kagglehub/competitions/{competition_name}/oil.csv"
]

combined_sources= "|".join(sources)

# Feature Set
standard_cols = ['unique_id', 'ds', 'y']
exogenous_features = [col for col in exo_train_df.columns if col not in standard_cols]


with mlflow.start_run(run_name="statsforecast/baseline+oil/2026-08-27") as run:

    # Log dataset metadata
    dataset = mlflow.data.from_pandas(
        exo_train_df,
        source=combined_sources,
        name='training_dataset_with_oil_exogenous'
    )
    mlflow.log_input(dataset, context="training_with_oil_exogenous")
    print("Dataset metadata logged successfully.")

    # Log architecture parameters
    mlflow.log_params(params_to_log)
    print("Logged parameters successfully.")


    # Log feature set
    mlflow.log_dict({"baseline_features": standard_cols, "exogenous_features": exogenous_features}, "feature_sets/features.json")
    print(f"Logged {len(exogenous_features)} exogenous features artifacts")
    print(f"Feature set logged successfully.")

    # Fit and forecast
    sf = StatsForecast(
        models=models,
        freq=freq, # Daily data
        n_jobs=-1, # Parallel
    )


    sf.save(path="statsforecast_models_oil")
    mlflow.log_artifacts("statsforecast_models_oil",
                         artifact_path="statsforecast_models_oil")

    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as tmp_file:
        joblib.dump(sf, tmp_file.name)
        mlflow.log_artifact(tmp_file.name, "model")


    # Use .forecast() for speed (no fitted values stored)
    forecasts = sf.forecast(
        df=exo_train_df,
        h=horizon,
        level=[conf_level], 
        X_df=exo_test_df[['unique_id', 'ds', 'oil_price']],
        n_jobs=-1
    )

    eval_df = exo_test_df.merge(forecasts.reset_index(), on=['unique_id', 'ds'])
    model_cols = [c for c in forecasts.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]
    results_df = evaluate_models(eval_df=eval_df, model_cols=model_cols)

    for _, row in results_df.iterrows():
        mlflow.log_metrics({
            f"{row['model']}_MAE": row["MAE"],
            f"{row['model']}_RMSE": row["RMSE"],
            f"{row['model']}_RMSLE": row["RMSLE"]
        })

    
    print("Generated forecasts with exogenous variable (oil price).")

## Evaluation

In [ ]:
eval_df

In [ ]:
# Test evaluation for a specific series
mask = eval_df['unique_id'] == '8_CLEANING'
print(mean_absolute_error(eval_df.loc[mask, 'y'], eval_df.loc[mask, 'AutoARIMA']))

In [ ]:
evaluate_models(eval_df, model_cols)
print_evaluation_results(results_df)

## Cross-Validation with Exogenous Variables

In [ ]:
df_oil = df_sample.merge(oil_df, how='left', left_on='ds', right_on='date')

In [ ]:
df_oil = df_oil.drop(columns=['date'])

In [ ]:
df_oil = df_oil.ffill().bfill()

In [ ]:
# StatForecast has built-in time series cross-validation
horizon = 16
step_size = 16
n_windows = 3
conf_level = 90

params_to_log_cv = {
    "forecast_horizon": horizon,
    "sf_models": model_names,
    "sf_freq": "D",
    "step_size": step_size,
    "conf_level": conf_level,
    "interval_n_windows": n_windows,
}

# Feature Set
standard_cols = ['unique_id', 'ds', 'y']
cols_to_log = standard_cols + ['oil_price']

sources = [
    f"/home/{os.getenv('USER')}/.cache/kagglehub/competitions/{competition_name}/train.csv",
    f"/home/{os.getenv('USER')}/.cache/kagglehub/competitions/{competition_name}/oil.csv"
]   

source = "|".join(sources)


with mlflow.start_run(run_name="statsforecast/crossval_oil/2026-09-10") as run:

    mlflow.log_params(params_to_log_cv)

    mlflow.log_dict({"baseline_features": standard_cols, "exogenous_features": ['oil_price']}, "feature_sets/features.json")

    mlflow.log_input(mlflow.data.from_pandas(df_oil, source=source, name='crossval_dataset_with_oil_exogenous'), context="crossval_with_oil_exogenous")

    

    crossval_exogen = sf.cross_validation(
        df=df_oil,
        h=horizon,
        step_size=step_size, # Non-overlapping windows
        n_windows=n_windows, # 3 folds
        level=[conf_level],
    )

    model_cols = [c for c in crossval_exogen.columns if not c in ['unique_id', 'ds', 'y', 'cutoff'] and 'lo' not in c and 'hi' not in c]

    for model in model_cols:
        mlflow.log_metrics({
            f"{model}_MAE": mean_absolute_error(crossval_exogen['y'], crossval_exogen['model']),
            f"{model}_RMSE": np.sqrt(mean_squared_error(crossval_exogen['y'], crossval_exogen['model'])),
            f"{model}_RMSLE": root_mean_squared_log_error(crossval_exogen['y'], crossval_exogen['model'])
        })

    print(crossval_exogen.head())

In [ ]:
model_cols = [c for c in forecasts.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]

In [ ]:
print("\n=== MAE ===")

for model in model_cols:
    mae = mean_absolute_error(crossval_exogen['y'], crossval_exogen[model])
    print(f"{model}: CV MAE: {mae:.2f}")

print("\n=== RMSE ===")

for model in model_cols:
    rmse = np.sqrt(mean_squared_error(crossval_exogen['y'], crossval_exogen[model]))
    print(f"{model}: CV RMSE:{rmse:.2f}")

## Visualization

In [ ]:
fig, axes = plt.subplots(len(subset_categories[:3]), 1, figsize=(14,10))

for ax, uid in zip(axes, subset_categories[:3]):

    # Historical
    hist = exo_train_df[exo_train_df['unique_id'] == uid].tail(60)
    ax2 = ax.twinx()
    ax2.plot(hist['ds'], hist['oil_price'], label='Oil Price', color='black')
    ax.plot(hist['ds'], hist['y'], label='Historical', color='gold')


    # Actual in test period
    actual = exo_test_df[exo_test_df['unique_id'] == uid]
    ax.plot(actual['ds'], actual['y'], label='Actual',  color='green', linewidth=2)
    ax2.plot(actual['ds'], actual['oil_price'], label='Oil_price',  color='black', linewidth=2)
    
    
    ax.set_title(uid)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('forecast_with_oil_price_comparison.png', dpi=150)
plt.show()
    

# NeuralForecast

## Setup

In [ ]:
import logging
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, RNN, NHITS, PatchTST

In [ ]:
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

## Model Training

In [ ]:
train_df.head(1)
val_df = train_df[train_df['ds'] > train_df['ds'].max() - pd.Timedelta(days=16)]
train_df = train_df[train_df['ds'] <= train_df['ds'].max() - pd.Timedelta(days=16)]

In [ ]:
%%capture

# Paramaters
horizon = 16
input_size = 2 * horizon # Input size for the neural network
freq = 'D' # Daily data
max_steps = 500
encoder_hidden_size = 64 # Number of cells in the encoder in LSTM, GRU, RNN
decoder_hidden_size = 64 # MLP number of neurons in each layer
scaler_type = 'standard' # StandardScaler or MinMaxScaler
n_windows = 2 # Number of windows for conformal intervals
intervals = ConformalIntervals(h=horizon, n_windows=n_windows) # TODO: How this works.

models = [
    LSTM(input_size=input_size,
         h=horizon,
         max_steps=max_steps,
         scaler_type=scaler_type,
         encoder_hidden_size=encoder_hidden_size,
         decoder_hidden_size=decoder_hidden_size,
        ),
    
    GRU(input_size=input_size,
         h=horizon,
         max_steps=max_steps,
         scaler_type=scaler_type,
         encoder_hidden_size=encoder_hidden_size,
         decoder_hidden_size=decoder_hidden_size,
        ),
    RNN(input_size=input_size,
         h=horizon,
         max_steps=max_steps,
         scaler_type=scaler_type,
         encoder_hidden_size=encoder_hidden_size,
         decoder_hidden_size=decoder_hidden_size,
        ),
    NHITS(input_size=input_size,
          h=horizon,
          max_steps=max_steps,
          scaler_type=scaler_type,
    ),
    PatchTST(input_size=input_size,
          h=horizon,
          max_steps=max_steps,
          scaler_type=scaler_type,
    ),  
    
]

model_names = [type(model).__name__ for model in models]

params_to_log = {
    "nf_horizon": horizon,
    "nf_input_size": input_size,
    "nf_max_steps": max_steps,
    "nf_models": model_names,
    "nf_encoder_hidden_size": encoder_hidden_size,
    "nf_decoder_hidden_size": decoder_hidden_size,
    "nf_scaler_type": scaler_type,
    "interval_type": "Conformal",
    "interval_n_windows": n_windows,
}


standard_cols = ['unique_id', 'ds', 'y']
exogenous_features = [col for col in train_df.columns if col not in standard_cols]


with mlflow.start_run(run_name="neuralforecast/baseline/2026-08-27") as run:
    
    # Log dataset metadata 
    dataset = mlflow.data.from_pandas(
        train_df,
        source=f"kaggle://competitions/{competition_name}/train.csv",
        name='training_dataset'
    )
    mlflow.log_input(dataset, context="training")
    print("Dataset metadata logged successfully.")

    # Log architecture parameters
    mlflow.log_params(params_to_log)
    print("Parameters logged successfully.")

    # Log feature set
    mlflow.log_dict({"baseline_features": standard_cols, "exogenous_features": exogenous_features}, "feature_sets/features.json")
    print(f"Logged {len(exogenous_features)} exogenous features artifacts")
    print(f"Feature set logged successfully.")

    nf = NeuralForecast(models=models, freq=freq)

    start_time = time.perf_counter()
    nf.fit(df=train_df, prediction_intervals=intervals, n_jobs=-1)

    end_time = time.perf_counter()
    print(f"Model fitting time: {end_time - start_time:.2f} seconds")
    mlflow.log_metric("training_time_in_seconds", end_time - start_time)

    nf.save(path="neuralforecast_models")
    mlflow.log_artifacts("neuralforecast_models",
        artifact_path="neuralforecast_models")
    
    with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as tmp_file:
        joblib.dump(nf, tmp_file.name)
        mlflow.log_artifact(tmp_file.name, "model")
        print(f"NeuralForecast model saved and logged to MLflow at {tmp_file.name}")



    print("NeuralForecast models fitted successfully.")

## Prediction and Evaluation

In [ ]:
with mlflow.start_run(run_name="neuralforecast/baseline_test/2026-09-10") as run:
    
    # Log dataset metadata 
    dataset = mlflow.data.from_pandas(
        val_df,
        source=f"/home/{os.getenv('USER')}/.cache/kagglehub/competitions/{competition_name}/test.csv",
        name='test_dataset'
    )
    mlflow.log_input(dataset, context="test")
    print("Dataset metadata logged successfully.")

    # Log architecture parameters
    mlflow.log_params(params_to_log)
    print("Parameters logged successfully.")

    # Log feature set
    mlflow.log_dict({"baseline_features": standard_cols, "exogenous_features": exogenous_features}, "feature_sets/features.json")
    print(f"Logged {len(exogenous_features)} exogenous features artifacts")
    print(f"Feature set logged successfully.")

    start_time = time.perf_counter()
    
    y_hat = nf.predict(df=val_df)

    end_time = time.perf_counter()
    print(f"Inference time: {end_time - start_time:.2f} seconds")

    model_cols = [c for c in y_hat.columns if not c in ['unique_id', 'ds'] and 'lo' not in c and 'hi' not in c]
    results_df = evaluate_models(eval_df, model_cols)

    for _, row in results_df.iterrows():
        mlflow.log_metrics({
            f"{row['model']}_MAE": row["MAE"],
            f"{row['model']}_RMSE": row["RMSE"],
            f"{row['model']}_RMSLE": row["RMSLE"]
        })


In [ ]:
y_hat.head(10)

In [ ]:

print_evaluation_results(results_df)

In [ ]:
fig, axes = plt.subplots(len(subset_categories[:3]), 1, figsize=(14,10))
for ax, uid in zip(axes, subset_categories[:3]):
    # Historical
    hist = train_df[train_df['unique_id'] == uid].tail(60)
    ax.plot(hist['ds'], hist['y'], label='Historical', color='black')

    # Actuals in test period
    actual = test_df[test_df['unique_id'] == uid]
    ax.plot(actual['ds'], actual['y'], label='Actual', color='green', linewidth=2)

    # Forecast (best 2 models)
    fc = y_hat.reset_index()
    fc_uid = fc[fc['unique_id'] == uid]
    ax.plot(fc_uid['ds'], fc_uid['LSTM'], label='LSTM', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['RNN'], label='RNN', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['GRU'], label='GRU', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['NHITS'], label='NHITS', linestyle='--')
    ax.plot(fc_uid['ds'], fc_uid['PatchTST'], label='PatchTST', linestyle='--')

    # Confidence interval
    if 'AutoARIMA-lo-90' in fc_uid.columns:
        ax.fill_between(fc_uid['ds'], fc_uid['AutoARIMA-lo-90'],
                        fc_uid['AutoARIMA-hi-90'], alpha=0.2)
    ax.set_title(uid)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150)
plt.show()